# Phenotype Objects

`snputils` uses phenotype containers to keep sample IDs aligned with trait values. This tutorial builds all inputs in memory, so it can be run without external files.

## Build a small phenotype table

`MultiPhenotypeObject` stores a table with the sample identifier in the first column and one or more phenotype columns after it. Use this object when you want to keep several traits together.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

RESULTS_DIR = Path("results/tutorials")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SEED = 20240520
from snputils.phenotype import MultiPhenotypeObject, PhenotypeObject, CovariateObject

rng = np.random.default_rng(SEED)
samples = np.array([f"S{i:02d}" for i in range(1, 9)])

phen_df = pd.DataFrame({
    "sample": samples,
    "height_z": np.round(rng.normal(size=samples.size), 3),
    "case_control": [0, 1, 0, 1, 0, 0, 1, 1],
    "batch": [1, 1, 2, 2, 1, 2, 1, 2],
})

multi = MultiPhenotypeObject(phen_df)
multi

MultiPhenotypeObject(shape=(8, 4), n_samples=8, n_phenotypes=3, sample_column='sample')

In [2]:
multi.phen_df

,sample,height_z,case_control,batch
0,S01,-0.194,0,1
1,S02,0.174,1,1
2,S03,0.911,0,2
3,S04,-0.234,1,2
4,S05,0.980,0,1
5,S06,0.483,0,2
6,S07,0.306,1,1
7,S08,-0.801,1,2


## Filter samples

Filtering returns a new object by default. `samples` selects by ID, `indexes` selects by row position, and `reorder=True` follows the requested order instead of preserving table order.

In [3]:
selected = multi.filter_samples(samples=["S05", "S02", "S01"], reorder=True)
selected.phen_df

,sample,height_z,case_control,batch
0,S05,0.980,0,1
1,S02,0.174,1,1
2,S01,-0.194,0,1


In [4]:
without_first_and_last = multi.filter_samples(indexes=[0, -1], include=False)
without_first_and_last.phen_df

,sample,height_z,case_control,batch
0,S02,0.174,1,1
1,S03,0.911,0,2
2,S04,-0.234,1,2
3,S05,0.980,0,1
4,S06,0.483,0,2
5,S07,0.306,1,1


## Single-trait phenotype objects

`PhenotypeObject` is the single-trait representation used by association workflows. Binary traits are normalized to 0/1; quantitative traits keep their numeric values.

In [5]:
binary = PhenotypeObject(
    samples=phen_df["sample"],
    values=phen_df["case_control"],
    phenotype_name="DISEASE",
    quantitative=False,
)

quantitative = PhenotypeObject(
    samples=phen_df["sample"],
    values=phen_df["height_z"],
    phenotype_name="HEIGHT_Z",
    quantitative=True,
)

print(binary)
print("cases:", binary.cases)
print("controls:", binary.controls)
print(quantitative)

PhenotypeObject(phenotype_name='DISEASE', shape=(8,), n_samples=8, trait_type='binary', n_cases=4, n_controls=4)
cases: ['S02', 'S04', 'S07', 'S08']
controls: ['S01', 'S03', 'S05', 'S06']
PhenotypeObject(phenotype_name='HEIGHT_Z', shape=(8,), n_samples=8, trait_type='quantitative')


## Covariates

`CovariateObject` stores a numeric sample-by-covariate matrix with explicit covariate names. Keeping covariates in a snputils object avoids separate path parsing in downstream tools.

In [6]:
covariates = CovariateObject(
    samples=phen_df["sample"],
    values=phen_df[["batch"]].to_numpy(),
    covariate_names=["batch"],
)

print(covariates)
pd.DataFrame(covariates.values, index=covariates.samples, columns=covariates.covariate_names)

CovariateObject(shape=(8, 1), n_samples=8, n_covariates=1)


,batch
S01,1.0
S02,1.0
S03,2.0
S04,2.0
S05,1.0
S06,2.0
S07,1.0
S08,2.0
